# EdgeVision v2 runner

Portable Colab runner for the repository. It is **not** an export of the original working notebook (linked in README.md). Reruns write to `*_rerun` directories so the preserved `results/` are not overwritten. Training is CPU-bound as originally run; a GPU runtime speeds up training only.

In [ ]:
!git clone https://github.com/umrzoq019/EdgeVision_research.git
%cd EdgeVision_research
!pip -q install -r requirements.txt

## Sanity check

In [ ]:
!pytest -q tests/

## 1. Baselines, 3 seeds (~18 min per seed on CPU)

In [ ]:
!python src/multi_seed.py --seeds 42 123 456 --output-root results/multi_seed_rerun

## 2. Distillation, 3 seeds (T=2, alpha=0.5)

In [ ]:
!python src/multi_seed_distill.py --seeds 42 123 456 --baseline-root results/multi_seed_rerun --output-root results/distillation/multi_seed_rerun

## 3. Distillation ablation (seed 42)

In [ ]:
!python src/distill_ablation.py --teacher results/multi_seed_rerun/seed_42/SmallCNNv2.pt --out-dir results/distillation/ablation_rerun

## 4. Interleaved latency benchmark (recommended over benchmark_repeated.py)

In [ ]:
!python src/benchmark_interleaved.py --checkpoint-root results/multi_seed_rerun --out results/latency/interleaved.json

## 5. INT8 post-training quantization

In [ ]:
!python src/quantize_ptq.py --checkpoint results/multi_seed_rerun/seed_42/NanoCNNv2.pt --model NanoCNNv2 --out-dir results/quantization_rerun

## 6. Full 10,000-image test evaluation

In [ ]:
!python src/eval_checkpoints.py --checkpoints results/multi_seed_rerun/seed_42/NanoCNNv2.pt results/distillation/multi_seed_rerun/seed_42/NanoCNNv2_distilled.pt --out results/analysis/full_test_eval_rerun.json

## 7. Tables, statistics, figures

In [ ]:
!python src/final_analysis.py --baseline-dir results/multi_seed_rerun --distill-dir results/distillation/multi_seed_rerun --out-dir results/analysis_rerun --fig-dir figures_rerun

## 8. Save to Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/EdgeVision && cp -r results results_*/ figures_rerun /content/drive/MyDrive/EdgeVision/ 2>/dev/null || true